In [ ]:
import os

# Change this variable if the root folder name has been changed
root_dir = "nvae-shape-encoding"
current_dir = os.getcwd()

if not current_dir.endswith(root_dir):
    %cd ../..

assert os.getcwd().endswith(root_dir)

In [ ]:
import lightning as L
import torch

from arch.simclr.utils import load_simclr_backbone
from const import SEED
from data_modules.acdc import ACDCMaskDataModule
from utils.utils import setup_device

model_path = "logs/simclr_acdc/resnet-18-v2-no-elastic/checkpoints/epoch=143-step=1008.ckpt"

# Setup device
device = setup_device()
print(f"Device: {device}")

# Seed
L.seed_everything(SEED)

# Load data
data_module = ACDCMaskDataModule(batch_size=20)

# Reseed after preprocessing data
L.seed_everything(SEED)

# Load model
model = load_simclr_backbone(model_path)

In [ ]:
loader_train = data_module.train_dataloader()

# Stack each batch
data_train = []

for batch in loader_train:
    data_train.append(batch)

data_train = torch.cat(data_train, dim=0)
print(data_train.shape)

loader_test = data_module.test_dataloader()

# Stack each batch
data_test = []

for batch in loader_test:
    data_test.append(batch)

data_test = torch.cat(data_test, dim=0)
print(data_test.shape)

In [ ]:
from utils.eval import encode_embeddings

model = model.to(device)

# Extract features for train images
train_feats = encode_embeddings(data_train, model, device)
print(train_feats.shape)

# Extract features for test images
test_feats = encode_embeddings(data_test, model, device)
print(test_feats.shape)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
pca.fit(train_feats)

print(f"Explained variance: {pca.explained_variance_ratio_}")
print(f"Sum: {pca.explained_variance_ratio_.sum()}")

train_feats_reduced = pca.transform(train_feats)
test_feats_reduced = pca.transform(test_feats)

print(train_feats_reduced.shape)
print(test_feats_reduced.shape)

In [ ]:
from matplotlib import pyplot as plt

plt.style.use("ggplot")

plt.scatter(train_feats_reduced[:, 0], train_feats_reduced[:, 1], alpha=0.5, s=10, label="Train")
plt.scatter(test_feats_reduced[:, 0], test_feats_reduced[:, 1], alpha=0.5, s=10, label="Test")

plt.tight_layout()
plt.legend()
plt.show()

In [ ]:
train_imgs = torch.argmax(data_train, dim=1).unsqueeze(1)
train_imgs = train_imgs.permute(0, 2, 3, 1).float()

test_imgs = torch.argmax(data_test, dim=1).unsqueeze(1)
test_imgs = test_imgs.permute(0, 2, 3, 1).float()

print(train_imgs.shape)
print(test_imgs.shape)

In [ ]:
import pandas as pd

train_df = pd.DataFrame(train_feats_reduced, columns=["x", "y"])
train_df["dataset"] = "Train"
train_df["image"] = [tensor.numpy() for tensor in train_imgs]

test_df = pd.DataFrame(test_feats_reduced, columns=["x", "y"])
test_df["dataset"] = "Test"
test_df["image"] = [tensor.numpy() for tensor in test_imgs]

df = pd.concat([train_df, test_df])
df = df.sample(n=100, replace=False)

print(df.shape)
print(df["image"].iloc[0].shape)
df.head()

In [ ]:
import altair as alt

data = df

selection = alt.selection_single(on="mouseover", nearest=True, clear="false")
scatter = alt.Chart(data).mark_circle().encode(
    alt.X('x:N',axis=None),
    alt.Y('y:N',axis=None),
    color=alt.condition(selection,
                        alt.value('lightgray'),
                        alt.Color('label:N')),
    # shape= alt.Shape('label:N', condition=selection,scale=alt.Scale(range=['circle','diamond'])), 
    size=alt.value(100),
    tooltip='label:N'
).add_selection(
    selection
).properties(
    width=400,
    height=400
)

digit  = alt.Chart(data).transform_filter(
    selection
).transform_window(
    index='count()'           # number each of the images
).transform_flatten(
    ['image']                 # extract rows from each image
).transform_window(
    row='count()',            # number the rows...
    groupby=['index']         # ...within each image
).transform_flatten(
    ['image']                 # extract the values from each row
).transform_window(
    column='count()',         # number the columns...
    groupby=['index', 'row']  # ...within each row & image
).mark_rect(stroke='black',strokeWidth=0).encode(
    alt.X('column:O', axis=None),
    alt.Y('row:O', axis=None),
    alt.Color('image:Q',sort='descending',
        scale=alt.Scale(scheme=alt.SchemeParams('lightgreyteal',
                        extent=[1, 0]),
        
        ),
        legend=None
    ),
).properties(
    width=400,
    height=400,
)

(scatter | digit).show()

In [ ]:
from arch.vae.vae import VAE
from utils.utils import discretise

def generate_data(model: VAE) -> torch.Tensor:
    num_samples = 512

    # Sample from latent space
    z = torch.randn(num_samples, model.hparams.latent_dim).to(device)

    with torch.no_grad():
        model.eval()
        model.to(device)
        
        # Generate segmentation maps from latent variables
        x_fake_logits: torch.Tensor = model.decoder(z)

    return x_fake_logits

vae_model_path = "logs/vae_acdc/info-vae/ld-6-beta-0-gamma-200/checkpoints/epoch=44-step=4815.ckpt"
vae_model = VAE.load_from_checkpoint(vae_model_path)

good_logits = generate_data(vae_model)

vae_model_path = "logs/vae_acdc/info-vae/ld-6-beta-0-gamma-5/checkpoints/epoch=48-step=5243.ckpt"
vae_model = VAE.load_from_checkpoint(vae_model_path)

bad_logits = generate_data(vae_model)

good_fake_data = discretise(good_logits)
bad_fake_data = discretise(bad_logits)

print(good_fake_data.shape)
print(bad_fake_data.shape)

In [ ]:
from utils.utils import show_samples

# View generations
good_generations = torch.argmax(good_logits, dim=1).unsqueeze(1)[:40]
show_samples(good_generations, rgb=False, ncol=10, figsize=(10, 4))

bad_generations = torch.argmax(bad_logits, dim=1).unsqueeze(1)[:40]
show_samples(bad_generations, rgb=False, ncol=10, figsize=(10, 4))

In [ ]:
good_feats = encode_embeddings(good_fake_data, model, device)
bad_feats = encode_embeddings(bad_fake_data, model, device)

good_feats_reduced = pca.transform(good_feats)
bad_feats_reduced = pca.transform(bad_feats)

print(good_feats_reduced.shape)
print(bad_feats_reduced.shape)

In [ ]:
plt.scatter(train_feats_reduced[:, 0], train_feats_reduced[:, 1], alpha=0.5, s=10, label="Train")
plt.scatter(good_feats_reduced[:, 0], good_feats_reduced[:, 1], alpha=0.5, s=10, label="Good")

plt.tight_layout()
plt.legend()
plt.show()

In [ ]:
plt.scatter(train_feats_reduced[:, 0], train_feats_reduced[:, 1], alpha=0.5, s=10, label="Train")
plt.scatter(bad_feats_reduced[:, 0], bad_feats_reduced[:, 1], alpha=0.5, s=10, label="Bad")

plt.tight_layout()
plt.legend()
plt.show()